# Agents and Chatbots
By: Thandokuhle Brian Msane\
_Credit: <a href="https://www.youtube.com/watch?v=jGg_1h0qzaM&"> YouTube Tutorial</a>_
## Agent I

The objectives of these simple bot are:
- define a state structure with a list of `HumanMessage` objects
- Initialize a `GPT-4o` model using `LangChain's ChatOpenAI`
- Sending and handling different types of messages
- Building and compiling the graph of the `Agent`

In [ ]:
import os
from typing import TypedDict, List, Union

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, START

load_dotenv()

True

In [ ]:
class AgentState(TypedDict):
    messages: List[HumanMessage]

llm = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=os.getenv("OPENAI_API_KEY")
)

def process(state: AgentState) -> AgentState:
    response = llm.invoke(input=state['messages'])
    print("AI: \n ", response)
    return state

graph = StateGraph(AgentState)
graph.add_node('process', process)
graph.add_edge(START, 'process')
graph.add_edge('process', END)

agent = graph.compile()

In [ ]:
user_content = input('Enter: ')

while user_content.lower() != 'exit':
    rsponse = agent.invoke(AgentState(messages=HumanMessage(user_content)))
    user_content = input('Enter: ')


## Chatbot
From the example above we notice that there is one major limitation which is that the LLM does not remember what we said. This is because we do not have memory so in this section we aim to fix that.

- use different message types - `HumanMessage` and `AIMessage`
- Maintain a full conversation history using both message types
- Create a sophisticated conversation loop


In [ ]:

class ChatbotState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]


def process(state: ChatbotState) -> ChatbotState:
    """Answer all your queries"""
    response = llm.invoke(state['messages'])
    state['messages'].append(AIMessage(content=response.content))
    return state

graph = StateGraph(ChatbotState)
graph.add_node('process', process)
graph.add_edge(START, 'process')
graph.add_edge('process', END)

agent = graph.compile()

conversation_history = []
user_input = input("Enter: ")

while user_input != "exit":
    conversation_history.append(HumanMessage(user_input))
    result = agent.invoke({'messages': conversation_history})
    print(result['messages'])
    conversation_history = result['messages']
    user_input = input("Enter: ")


Observation

The key problem with this part is that as we keep conversing with the agent, the list of Human and AI messages in the state increases. With this, we use many tokens hence a lot of money. One solution to this is discarding some messages that we think are no longer useful. Another solution could be keeping only a summary of the conversation instead of every work.

Another problem is that we lost the conversation history when we restart the code. We might need to use a vector database to store the context and retrieve it later, following the RAG approach.